# DINO-WM: PointMaze, our own training run
Train our own PointMaze world model to **15 epochs** on free Colab T4s, then measure **success rate vs. epoch** (Section 4) and compare with the paper's **98%**.

Our changes to the authors' code (all patched on Drive, applied in Section 1, safe to rerun):
- **fp16 training step** (`ARPL_AMP=1`): same losses as fp32, 2.2× faster
- **fused attention** (`ARPL_SDPA=1`): same outputs, no peeking at later frames, 1.7× faster
- **per-epoch loss log** `outputs/point_maze_2s/epoch_losses.csv`, saved even if the runtime dies
- **resume fix**: the original code re-created the action and proprio encoders with random weights on every resume; now they are kept
- full logging to Drive (`ARPL-WM/tools/arpl_logger.py`, started by the training cell): every session's full output, GPU use every minute, one row per session (GPU, settings, epoch it resumed from) and one per epoch (minutes, speed, losses) in `ARPL-WM/logs/point_maze_2s/`
- plus the earlier fixes: DINOv2 pin, memory-mapped loader, validation without gradients, chunked CEM
- **no decoder** (only makes pictures; never affects the predictor or planning)

| Every session | |
|---|---|
| 1 | Runtime → T4 GPU, run **Section 1** top to bottom |
| 2 | Run the **training cell** in Section 3: it resumes from the last saved epoch and stops at 15 |
| 3 | After training (or in a session without training): **Section 4** evaluation |

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%env WM=/content/drive/MyDrive/ARPL-WM
%env PY=/content/mc/envs/dino_wm/bin/python
# Add MuJoCo's library to the search path WITHOUT removing Colab's NVIDIA driver path
# (replacing it hides the GPU: "GPU visible: False")
import os
os.environ['LD_LIBRARY_PATH'] = os.environ.get('LD_LIBRARY_PATH', '') + ':/root/.mujoco/mujoco210/bin:/usr/lib/nvidia'
# Render simulations without a screen
%env MUJOCO_GL=egl
# Log training metrics locally instead of to wandb.ai
%env WANDB_MODE=offline
# Where the repo looks for datasets
%env DATASET_DIR=/content/data
# Print output immediately (needed so logs fill up live)
%env PYTHONUNBUFFERED=1
# Hide the harmless mjrl/flow/carla warnings from d4rl
%env D4RL_SUPPRESS_IMPORT_ERROR=1

# Create the result folders on Drive (does nothing if they already exist)
!mkdir -p $WM/outputs $WM/plan_outputs $WM/data_zips $WM/logs

In [ ]:
# OpenGL/OSMesa libraries that mujoco-py compiles against
!apt-get update -qq
!apt-get install -y -qq libosmesa6-dev libgl1-mesa-dev libglew-dev patchelf
# MuJoCo 2.1 physics engine (wiped on every restart, so download each session)
![ -d ~/.mujoco/mujoco210 ] || (mkdir -p ~/.mujoco && wget -q https://mujoco.org/download/mujoco210-linux-x86_64.tar.gz -O - | tar -xz -C ~/.mujoco)
!ls ~/.mujoco/mujoco210/bin | head -3

In [ ]:
# Install Miniconda (skipped if already installed in this session)
![ -d /content/mc ] || (wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O mc.sh && bash mc.sh -b -p /content/mc)
!/content/mc/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/content/mc/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
# Copy the code into your Drive folder once. Later sessions skip this and use your edited version.
![ -d $WM/dino_wm ] || git clone https://github.com/gaoyuezhou/dino_wm.git $WM/dino_wm

In [ ]:
# Unpack the saved environment from Drive, only if this session doesn't have it yet
![ -x $PY ] || (mkdir -p /content/mc/envs && tar -xzf $WM/dino_wm_env.tgz -C /content/mc/envs)
# All libraries the code needs. Should print "imports ok"
!$PY -c "import d4rl, mujoco_py, gym, hydra, decord, einops, accelerate, wandb, pymunk, shapely, torch; print('imports ok')"
# GPU check. Should print the GPU name and "GPU visible: True"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!$PY -c "import torch; print('GPU visible:', torch.cuda.is_available())"

In [ ]:
# Pin DINOv2 to its Oct 2024 version (Python 3.9 compatible; see DINO-WM issue #25). Safe to rerun.
!sed -i 's|torch.hub.load("facebookresearch/dinov2", name)|torch.hub.load("facebookresearch/dinov2:85a24602099d397264d5b30461ad7f3bfd726ca1", name)|' $WM/dino_wm/models/dino.py
# Should show the line with "dinov2:85a2460..."
!grep -n "torch.hub.load" $WM/dino_wm/models/dino.py

In [ ]:
# DINOv2 ViT-S/14 weights without Meta's server: dl.fbaipublicfiles.com has an expired certificate (since Nov 11, 2026).
# First session: takes the same Meta weights from Hugging Face (facebook/dinov2-small), renames the keys back to Meta's
# format, checks they load exactly into the DINOv2 model and give Hugging Face's features, and saves the file on Drive.
# Every later session: only copies the Drive file into torch's cache (instant). Runs in Colab's own Python.
import os, shutil, torch
WM = os.environ['WM']
dst = f'{WM}/cache/dinov2_vits14_pretrain.pth'
hub = '/root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth'
os.makedirs(os.path.dirname(dst), exist_ok=True); os.makedirs(os.path.dirname(hub), exist_ok=True)
if not os.path.exists(dst):
    from safetensors.torch import load_file
    from huggingface_hub import hf_hub_download
    hf = load_file(hf_hub_download('facebook/dinov2-small', 'model.safetensors'))
    sd = {'cls_token': hf['embeddings.cls_token'], 'pos_embed': hf['embeddings.position_embeddings'],
          'mask_token': hf['embeddings.mask_token'],
          'patch_embed.proj.weight': hf['embeddings.patch_embeddings.projection.weight'],
          'patch_embed.proj.bias': hf['embeddings.patch_embeddings.projection.bias'],
          'norm.weight': hf['layernorm.weight'], 'norm.bias': hf['layernorm.bias']}
    depth = 1 + max(int(k.split('.')[2]) for k in hf if k.startswith('encoder.layer.'))
    for i in range(depth):
        h, m = f'encoder.layer.{i}.', f'blocks.{i}.'
        a = h + 'attention.attention.'
        for t in ('weight', 'bias'):
            sd[m + 'attn.qkv.' + t] = torch.cat([hf[a + 'query.' + t], hf[a + 'key.' + t], hf[a + 'value.' + t]])
            sd[m + 'attn.proj.' + t] = hf[h + 'attention.output.dense.' + t]
            for n in ('norm1', 'norm2', 'mlp.fc1', 'mlp.fc2'):
                sd[m + n + '.' + t] = hf[h + n + '.' + t]
        sd[m + 'ls1.gamma'] = hf[h + 'layer_scale1.lambda1']
        sd[m + 'ls2.gamma'] = hf[h + 'layer_scale2.lambda1']
    # check 1: loads into the pinned DINOv2 model with no missing, extra or misshaped weights
    model = torch.hub.load('facebookresearch/dinov2:85a24602099d397264d5b30461ad7f3bfd726ca1', 'dinov2_vits14',
                           pretrained=False, trust_repo=True, skip_validation=True).eval()  # skip GitHub's commit check (rate-limited)
    model.load_state_dict(sd, strict=True)
    # check 2: same patch features as Hugging Face's model (518x518 = native size, so no resizing differences)
    from transformers import Dinov2Model
    ref = Dinov2Model.from_pretrained('facebook/dinov2-small').eval()
    x = torch.randn(1, 3, 518, 518)
    with torch.no_grad():
        diff = (model.forward_features(x)['x_norm_patchtokens'] - ref(pixel_values=x).last_hidden_state[:, 1:]).abs().max().item()
    print(f'max feature difference vs Hugging Face: {diff:.2e}')
    assert diff < 1e-3, 'conversion mismatch'
    torch.save(sd, dst)
shutil.copy(dst, hub)
print('DINOv2 weights ready:', hub, f'{os.path.getsize(hub) / 1e6:.0f} MB')

In [ ]:
# Unzip the PointMaze dataset to local disk, only if it isn't there yet
![ -d /content/data/point_maze ] || (mkdir -p /content/data && cd /content/data && unzip -qo $WM/data_zips/point_maze.zip)
!ls /content/data/point_maze

In [ ]:
%%writefile /content/patch_amp.py
# [ARPL-WM] Adds an optional fp16 mixed-precision mode to train.py (off unless ARPL_AMP=1). Safe to rerun.
import sys, shutil, os
path = sys.argv[1]
src = open(path).read()
if "[ARPL-WM patch] fp16" in src:
    print("already patched"); sys.exit(0)
if not os.path.exists(path + ".amp.orig"):
    shutil.copy(path, path + ".amp.orig")
reps = [
("from accelerate import Accelerator\n",
 "from accelerate import Accelerator\n"
 "# [ARPL-WM patch] fp16: optional mixed precision for the training step only (ARPL_AMP=1). Validation stays fp32.\n"
 "_ARPL_AMP = os.environ.get(\"ARPL_AMP\", \"0\") == \"1\"\n"
 "def _arpl_opt_step(trainer, opt):\n"
 "    if _ARPL_AMP:\n"
 "        trainer._arpl_scaler.step(getattr(opt, \"optimizer\", opt))\n"
 "    else:\n"
 "        opt.step()\n"),
("""    def train(self):
        for i, data in enumerate(""",
"""    def train(self):
        if _ARPL_AMP and not hasattr(self, "_arpl_scaler"):  # [ARPL-WM patch] fp16
            self._arpl_scaler = torch.cuda.amp.GradScaler()
            log.info("[ARPL-WM] fp16 mixed precision ON for training")
        for i, data in enumerate("""),
("""            self.model.train()
            z_out, visual_out, visual_reconstructed, loss, loss_components = self.model(
                obs, act
            )
""",
"""            self.model.train()
            with torch.autocast("cuda", dtype=torch.float16, enabled=_ARPL_AMP):  # [ARPL-WM patch] fp16
                z_out, visual_out, visual_reconstructed, loss, loss_components = self.model(
                    obs, act
                )
"""),
("""            self.accelerator.backward(loss)

            if self.model.train_encoder:
                self.encoder_optimizer.step()
            if self.cfg.has_decoder and self.model.train_decoder:
                self.decoder_optimizer.step()
            if self.cfg.has_predictor and self.model.train_predictor:
                self.predictor_optimizer.step()
                self.action_encoder_optimizer.step()
""",
"""            if _ARPL_AMP:  # [ARPL-WM patch] fp16: scaled backward, so small fp16 gradients don't round to zero
                self._arpl_scaler.scale(loss).backward()
            else:
                self.accelerator.backward(loss)

            if self.model.train_encoder:
                _arpl_opt_step(self, self.encoder_optimizer)
            if self.cfg.has_decoder and self.model.train_decoder:
                _arpl_opt_step(self, self.decoder_optimizer)
            if self.cfg.has_predictor and self.model.train_predictor:
                _arpl_opt_step(self, self.predictor_optimizer)
                _arpl_opt_step(self, self.action_encoder_optimizer)
            if _ARPL_AMP:
                self._arpl_scaler.update()
"""),
]
for old, new in reps:
    assert src.count(old) == 1, "pattern not found exactly once:\n" + old[:80]
    src = src.replace(old, new)
open(path, "w").write(src)
print("patched OK")


In [ ]:
# Apply the fp16 patch to train.py (keeps the original as train.py.amp.orig). Does nothing when ARPL_AMP is not "1".
!$PY /content/patch_amp.py $WM/dino_wm/train.py
!grep -c "ARPL-WM patch\] fp16" $WM/dino_wm/train.py

In [ ]:
%%writefile /content/patch_epochlog.py
# [ARPL-WM] After every epoch, append its losses to epoch_losses.csv in the run folder and close the file right away,
# so Google Drive saves it even if the runtime is killed later. Safe to rerun.
import sys, shutil, os
path = sys.argv[1]
src = open(path).read()
if "[ARPL-WM patch] epochlog" in src:
    print("already patched"); sys.exit(0)
if not os.path.exists(path + ".epochlog.orig"):
    shutil.copy(path, path + ".epochlog.orig")
old = """        if self.accelerator.is_main_process:
            self.wandb_run.log(epoch_log)
"""
new = """        if self.accelerator.is_main_process:  # [ARPL-WM patch] epochlog: one line per epoch, file closed immediately
            _new_file = not os.path.exists("epoch_losses.csv")
            with open("epoch_losses.csv", "a") as _fh:
                if _new_file:
                    _fh.write(",".join(epoch_log.keys()) + "\\n")
                _fh.write(",".join(f"{float(v):.6f}" for v in epoch_log.values()) + "\\n")
                _fh.flush(); os.fsync(_fh.fileno())
        if self.accelerator.is_main_process:
            self.wandb_run.log(epoch_log)
"""
assert src.count(old) == 1, "pattern not found exactly once"
open(path, "w").write(src.replace(old, new))
print("patched OK")


In [ ]:
# Apply the per-epoch loss log patch to train.py (keeps the original as train.py.epochlog.orig)
!$PY /content/patch_epochlog.py $WM/dino_wm/train.py
!grep -c "ARPL-WM patch\] epochlog" $WM/dino_wm/train.py

In [ ]:
%%writefile /content/patch_resume.py
# [ARPL-WM] Fix resume: train.py re-created the action and proprio encoders with fresh random weights even when they
# were just loaded from the checkpoint. Now they are created only if the checkpoint did not provide them. Safe to rerun.
import sys, shutil, os
path = sys.argv[1]
src = open(path).read()
if "[ARPL-WM patch] resume" in src:
    print("already patched"); sys.exit(0)
if not os.path.exists(path + ".resume.orig"):
    shutil.copy(path, path + ".resume.orig")
reps = [
("""        self.proprio_encoder = hydra.utils.instantiate(
            self.cfg.proprio_encoder,
            in_chans=self.datasets["train"].proprio_dim,
            emb_dim=self.cfg.proprio_emb_dim,
        )
""",
"""        if self.proprio_encoder is None:  # [ARPL-WM patch] resume: keep the encoder loaded from the checkpoint
            self.proprio_encoder = hydra.utils.instantiate(
                self.cfg.proprio_encoder,
                in_chans=self.datasets["train"].proprio_dim,
                emb_dim=self.cfg.proprio_emb_dim,
            )
"""),
("""        self.action_encoder = hydra.utils.instantiate(
            self.cfg.action_encoder,
            in_chans=self.datasets["train"].action_dim,
            emb_dim=self.cfg.action_emb_dim,
        )
""",
"""        if self.action_encoder is None:  # [ARPL-WM patch] resume: keep the encoder loaded from the checkpoint
            self.action_encoder = hydra.utils.instantiate(
                self.cfg.action_encoder,
                in_chans=self.datasets["train"].action_dim,
                emb_dim=self.cfg.action_emb_dim,
            )
"""),
]
for old, new in reps:
    assert src.count(old) == 1, "pattern not found exactly once:\n" + old[:80]
    src = src.replace(old, new)
open(path, "w").write(src)
print("patched OK")


In [ ]:
# Apply the resume fix to train.py (keeps the original as train.py.resume.orig)
!$PY /content/patch_resume.py $WM/dino_wm/train.py
!grep -c "ARPL-WM patch\] resume" $WM/dino_wm/train.py

In [ ]:
%%writefile /content/patch_sdpa.py
# [ARPL-WM] Optional fused attention (PyTorch scaled_dot_product_attention) in the predictor, on when ARPL_SDPA=1. Safe to rerun.
import sys, shutil, os
path = sys.argv[1]
src = open(path).read()
if "[ARPL-WM patch] sdpa" in src:
    print("already patched"); sys.exit(0)
if not os.path.exists(path + ".sdpa.orig"):
    shutil.copy(path, path + ".sdpa.orig")
reps = [
("import torch\nfrom torch import nn\n",
 "import os\nimport torch\nfrom torch import nn\n"),
("""        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        # apply causal mask
        dots = dots.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))

        attn = self.attend(dots)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
""",
"""        if os.environ.get("ARPL_SDPA", "0") == "1":  # [ARPL-WM patch] sdpa: same math in one fused kernel
            out = torch.nn.functional.scaled_dot_product_attention(
                q, k, v,
                attn_mask=(self.bias[:, :, :T, :T] != 0),  # True = may attend (same causal frame mask)
                dropout_p=self.dropout.p if self.training else 0.0,
                scale=self.scale,
            )
        else:
            dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
            # apply causal mask
            dots = dots.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))

            attn = self.attend(dots)
            attn = self.dropout(attn)

            out = torch.matmul(attn, v)
"""),
]
for old, new in reps:
    assert src.count(old) == 1, "pattern not found exactly once:\n" + old[:80]
    src = src.replace(old, new)
open(path, "w").write(src)
print("patched OK")


In [ ]:
# Apply the fused-attention patch (keeps the original as vit.py.sdpa.orig). Does nothing unless ARPL_SDPA is "1".
!$PY /content/patch_sdpa.py $WM/dino_wm/models/vit.py
!grep -c "ARPL-WM patch\] sdpa" $WM/dino_wm/models/vit.py

## 3. Train up to 15 epochs (any number of sessions; rerun the same cell to resume)
Settings from Section 2: fp16 (`AMP`), batch 32 like the authors (`BATCH`), fused attention (`SDPA`), 2 workers. **≈60 min per epoch** (about 4 epochs per free session). Going past the authors' 10 shows whether their stopping point was reasonable (does success keep improving after epoch 10?).

**Every new session:** Section 1 (setup + data), then this cell. Nothing else. It finds `outputs/point_maze_2s/checkpoints/model_latest.pth` and continues from the next epoch (the model and epoch number are restored; Adam's running averages and the fp16 scale restart, which is how the original code resumes too). A disconnect loses at most the epoch in progress. Keep the settings below unchanged between sessions, and keep the Colab tab open while it trains.

Each checkpoint is about 0.25 GB on Drive (≈4 GB for 15).

In [ ]:
import os, re, time
AMP = "1"     # fp16 (our patch). Measured: same losses as fp32, 2.2x faster
BATCH = 32    # same as the authors (fits with fp16)
SDPA = "1"    # fused attention (our patch). Measured: same outputs, 1.7x faster, no peeking at later frames
TARGET_EPOCH = 15             # train until this epoch exists, then stop (works for any resume)
RUN_NAME = "point_maze_2s"    # folder in ARPL-WM/outputs/
SESSION = time.strftime("%Y%m%d_%H%M", time.gmtime())  # this session's id (UTC start time), names its log files
ck = f"/content/drive/MyDrive/ARPL-WM/outputs/{RUN_NAME}/checkpoints"
done = max([int(m.group(1)) for f in (os.listdir(ck) if os.path.isdir(ck) else []) if (m := re.fullmatch(r"model_(\d+)\.pth", f))], default=0)
remaining = TARGET_EPOCH - done
print(f"{RUN_NAME}: last saved epoch {done}, training {max(remaining, 0)} more to reach {TARGET_EPOCH}")
# values go to the shell as environment variables ({...} is not filled in on ! lines that contain $PY)
os.environ.update(ARPL_AMP=AMP, ARPL_SDPA=SDPA, ARPL_BATCH=str(BATCH), ARPL_EPOCHS=str(remaining), ARPL_RUN=RUN_NAME, ARPL_SESSION=SESSION)
if remaining > 0:
    # Background logger (ARPL-WM/tools/arpl_logger.py). Every minute it saves to ARPL-WM/logs/<RUN_NAME>/: this session's full output,
    # GPU use, a row in sessions.csv (GPU, settings, resumed from) and a row in epoch_timing.csv per finished epoch. Replaces an older logger.
    !cd /content && nohup python3 $WM/tools/arpl_logger.py $ARPL_RUN /content/train_$ARPL_SESSION.log $ARPL_SESSION > /content/arpl_logger.out 2>&1 &
    !cd $WM/dino_wm && $PY train.py env=point_maze frameskip=5 num_hist=3 env.num_workers=2 training.batch_size=$ARPL_BATCH has_decoder=False model.train_decoder=False training.epochs=$ARPL_EPOCHS training.save_every_x_epoch=1 ckpt_base_path=$WM hydra.run.dir=$WM/outputs/$ARPL_RUN 2>&1 | tee /content/train_$ARPL_SESSION.log
    # final copy of the full output (the logger also copies it every minute while training runs)
    !mkdir -p $WM/logs/$ARPL_RUN && cp /content/train_$ARPL_SESSION.log $WM/logs/$ARPL_RUN/session_${ARPL_SESSION}_stdout.log

In [ ]:
# Everything logged so far. All of it is on Drive: ARPL-WM/outputs/point_maze_2s/ and ARPL-WM/logs/point_maze_2s/
!ls -lh $WM/outputs/point_maze_2s/checkpoints/ 2>/dev/null || echo "no checkpoints yet"
# Epochs 1 to 6, rebuilt from checkpoint save times and earlier outputs (before the logger existed)
!cat $WM/logs/point_maze_2s/epoch_history_before_logger.csv
# All losses of every epoch from epoch 6 on (train.py writes one line per epoch)
!cat $WM/outputs/point_maze_2s/epoch_losses.csv 2>/dev/null || echo "no epoch_losses.csv yet"
# One row per session (GPU, settings, epoch it resumed from) and one per epoch (minutes, speed, losses), from the logger
!cat $WM/logs/point_maze_2s/sessions.csv; echo; cat $WM/logs/point_maze_2s/epoch_timing.csv 2>/dev/null || echo "no finished epoch logged yet"
# Latest GPU samples (utilization %, memory MiB, temperature C, power W) of the newest session
!tail -3 $(ls -t $WM/logs/point_maze_2s/gpu_*.csv | head -1)

### 3b. Validation loss of one saved checkpoint (about 8 min on a T4)
Recomputes the validation loss of `model_<VAL_EPOCH>.pth` exactly as `train.py` does at the end of an epoch (same split, fp32, dropout off), in a scratch folder. Needs the resume fix from Section 1, otherwise the action and proprio encoders are random and the number is wrong. Used to recover epoch 2 (0.0104).

In [ ]:
%%writefile /content/val_ckpt.py
# [ARPL-WM] Validation loss of one saved checkpoint, computed exactly as train.py does at the end of an epoch
# (same config, same 90/10 split from the same seed, plain fp32, original attention, dropout off).
# Usage: python val_ckpt.py <repo dir> <run dir> <epoch>
import os, sys, tempfile
from collections import OrderedDict
import torch
from omegaconf import OmegaConf

repo, run_dir, epoch = sys.argv[1], sys.argv[2], int(sys.argv[3])
ckpt = os.path.join(run_dir, "checkpoints", f"model_{epoch}.pth")
assert os.path.exists(ckpt), f"no checkpoint {ckpt}"

if not torch.cuda.is_available():  # CPU fallback: checkpoints hold GPU tensors
    _load = torch.load
    torch.load = lambda *a, **k: _load(*a, **{**k, "map_location": k.get("map_location", "cpu")})

os.environ.pop("ARPL_AMP", None)   # validation is fp32 anyway
if os.environ.get("ARPL_VAL_SDPA") == "1":   # fused attention, as during training-time validation
    os.environ["ARPL_SDPA"] = "1"
else:                                        # authors' original attention code
    os.environ.pop("ARPL_SDPA", None)
print("attention:", "fused (ARPL_SDPA=1)" if os.environ.get("ARPL_SDPA") == "1" else "original")

cfg = OmegaConf.load(os.path.join(run_dir, "hydra.yaml"))
work = tempfile.mkdtemp(prefix=f"val_ep{epoch}_")      # scratch run folder, nothing is written to the real one
os.makedirs(os.path.join(work, "checkpoints"))
os.symlink(ckpt, os.path.join(work, "checkpoints", "model_latest.pth"))
cfg.saved_folder = work
os.chdir(work)
sys.path.insert(0, repo)
import train

class _NotMultirun:  # train.py asks Hydra whether it is a multirun; we are not running under @hydra.main
    mode = train.RunMode.RUN
train.HydraConfig.get = lambda: _NotMultirun

trainer = train.Trainer(cfg)
assert trainer.epoch == epoch, f"loaded epoch {trainer.epoch}, expected {epoch}"
trainer.train_traj_dset = []   # skip the random open-loop rollout plots; they are not part of val_loss
trainer.epoch_log = OrderedDict()
trainer.val()
count, total = trainer.epoch_log["val_loss"]
print(f"\nRESULT epoch {epoch}: validation loss {total / count:.4f}  ({count} batches)")
for k in ("val_z_loss", "val_z_visual_loss", "val_z_proprio_loss"):
    if k in trainer.epoch_log:
        c, t = trainer.epoch_log[k]; print(f"  {k}: {t / c:.4f}")


In [ ]:
# Validation loss of one checkpoint. Epoch number goes to the shell as an environment variable
import os
VAL_EPOCH = 3   # sanity check: training printed 0.0090 for epoch 3
os.environ['ARPL_VAL_EPOCH'] = str(VAL_EPOCH)
!$PY /content/val_ckpt.py $WM/dino_wm $WM/outputs/point_maze_2s $ARPL_VAL_EPOCH 2>&1 | tee $WM/logs/val_ep${ARPL_VAL_EPOCH}.log
!grep -a "RESULT\|val_z" $WM/logs/val_ep${ARPL_VAL_EPOCH}.log

## 4. Evaluation: success rate
Same planner as the repo's `plan_point_maze.yaml`: CEM with 300 samples, top 30, 10 steps, horizon 5 (= 25 real steps), `alpha=0`, MPC executes the 5 planned steps and replans until every episode succeeds. Only difference: a cap of `MAX_ROUNDS` replanning rounds so a stuck episode can't loop forever. Success after **every** round is read from the log, so each run gives both "success within 1 round (25 steps)" and "success within all rounds".

- **4.0** settings + helper (run first, instant)
- **4a** success-vs-epoch curve: 10 episodes (seed 99) per checkpoint
- **4b** paper comparison: **50 episodes** for the final checkpoint, run as 5 parts of 10 (seeds 99 to 103, seed 99 is shared with 4a). Each part is saved separately, so a disconnect loses at most one part (about 35 to 40 min per round on a T4)
- **4c** results table + plot, next to the paper's **98%** for DINO-WM on PointMaze

Finished parts are skipped on rerun. Planning uses fused attention (same math, faster).

In [ ]:
# 4.0 Evaluation settings and helper (instant)
import os, re, json
WM = '/content/drive/MyDrive/ARPL-WM'
os.environ['ARPL_SDPA'] = '1'              # fused attention in planning: same math, faster
MAX_ROUNDS = 3                             # MPC replanning cap (repo: unlimited). 3 rounds = 75 real steps. Raise to 5 for a stricter paper match (slower for weak checkpoints)
CURVE_EPOCHS = [1, 3, 6, 10]           # 4a: 10 episodes each. Early epochs may need all rounds (about 35-40 min per round)
FINAL_EPOCHS = [10]                        # 4b: 50 episodes each
SEEDS_50 = [99, 100, 101, 102, 103]        # 5 parts x 10 episodes = 50

CKPT = f'{WM}/outputs/point_maze_2s/checkpoints'
def saved_epochs():
    return sorted(int(m.group(1)) for f in os.listdir(CKPT) if (m := re.fullmatch(r'model_(\d+)\.pth', f)))

def out_dir(epoch, seed):
    return f'{WM}/plan_outputs/point_maze_2s_ep{epoch}_s{seed}_r{MAX_ROUNDS}'

def finished(out):
    f = f'{out}/logs.json'
    return os.path.exists(f) and 'final_eval/success_rate' in open(f).read()

def evaluate(epoch, seed):
    out = out_dir(epoch, seed)
    if finished(out):
        print(f'epoch {epoch}, seed {seed}: already done, skipping'); return
    if epoch not in saved_epochs():
        print(f'epoch {epoch}: no checkpoint yet, skipping'); return
    print(f'=== epoch {epoch}, seed {seed}, 10 episodes, up to {MAX_ROUNDS} rounds ===')
    # values go to the shell as environment variables ({...} is not filled in on ! lines that contain $PY)
    os.environ.update(ARPL_EP=str(epoch), ARPL_SEED=str(seed), ARPL_OUT=out, ARPL_ROUNDS=str(MAX_ROUNDS))
    !cd $WM/dino_wm && $PY plan.py --config-name plan_point_maze.yaml ckpt_base_path=$WM model_name=point_maze_2s model_epoch=$ARPL_EP n_evals=10 seed=$ARPL_SEED planner.max_iter=$ARPL_ROUNDS hydra.run.dir=$ARPL_OUT 2>&1 | tee $ARPL_OUT.log
print('saved checkpoints:', saved_epochs())

In [ ]:
# 4a Success vs. epoch: 10 episodes (seed 99) per checkpoint
for e in CURVE_EPOCHS:
    evaluate(e, 99)

In [ ]:
# 4b Paper comparison: 50 episodes (5 parts x 10) for the final checkpoint(s)
for e in FINAL_EPOCHS:
    for s in SEEDS_50:
        evaluate(e, s)

In [ ]:
# 4c Results: success within round 1 (25 steps) and within all rounds, per epoch; 50-episode total vs. the paper
import glob, math
import matplotlib.pyplot as plt
def read_run(out):
    """-> (successes after each round, episodes, final success rate) or None"""
    if not finished(out): return None
    log = open(out + '.log', errors='ignore').read() if os.path.exists(out + '.log') else ''
    rounds = [blk.count('True') for blk in re.findall(r'self\.is_success:\s*\[([^\]]*)\]', log, re.S)]
    n = 10
    final = [json.loads(l) for l in open(f'{out}/logs.json') if 'final_eval/success_rate' in l][-1]['final_eval/success_rate']
    return rounds, n, final
print(f"Success vs. epoch (10 episodes, seed 99, up to {MAX_ROUNDS} rounds)")
print(f"{'epoch':>5} {'round 1':>8} {'all rounds':>11}")
curve = []
for e in sorted(set(CURVE_EPOCHS + FINAL_EPOCHS)):
    r = read_run(out_dir(e, 99))
    if r is None: continue
    rounds, n, final = r
    r1 = rounds[0] / n if rounds else float('nan')
    curve.append((e, r1, final)); print(f"{e:>5} {r1:>8.2f} {final:>11.2f}")
for e in FINAL_EPOCHS:
    parts = [read_run(out_dir(e, s)) for s in SEEDS_50]
    done = [p for p in parts if p]
    if not done: continue
    n = sum(p[1] for p in done); k_all = round(sum(p[2] * p[1] for p in done)); k1 = sum(p[0][0] for p in done if p[0])
    se = math.sqrt(max(k_all / n * (1 - k_all / n), 1e-9) / n)
    print(f"\nEpoch {e}, {n} episodes ({len(done)}/5 parts): round 1 {k1}/{n} = {k1/n:.2f}, "
          f"all rounds {k_all}/{n} = {k_all/n:.2f} (+/- {2*se:.2f}, 95%)   paper: 0.98")
if curve:
    plt.plot([c[0] for c in curve], [c[2] for c in curve], 'o-', label=f'within {MAX_ROUNDS} rounds')
    plt.plot([c[0] for c in curve], [c[1] for c in curve], 's--', label='within round 1 (25 steps)')
    plt.axhline(0.98, ls=':', c='gray', label='paper (DINO-WM, 50 episodes)')
    plt.xlabel('training epoch'); plt.ylabel('success rate (10 episodes)'); plt.ylim(-0.05, 1.05)
    plt.legend(); plt.title('PointMaze: success vs. training'); plt.show()

## Fixes (only when needed)
Run these only if `imports ok` does not print in Setup.

In [ ]:
# If "imports ok" did not print: reinstall missing pip packages from the repo's environment.yaml
!cd $WM/dino_wm && sed -n '/- pip:/,$p' environment.yaml | grep '==' | sed 's/^ *- //' > /content/pip_reqs.txt
!$PY -m pip install -r /content/pip_reqs.txt

In [ ]:
# After a fix: save the repaired environment to Drive (takes several minutes)
!tar -czf /content/dino_wm_env_new.tgz -C /content/mc/envs dino_wm && cp /content/dino_wm_env_new.tgz $WM/dino_wm_env.tgz && echo SAVED_OK